# Task #2: Reshape QPASS observations and check data quality

This notebook now uses Dylan's task #1 loader for all workbooks in `data/raw/QPASS/`, then calls the task #2 functions in `src/qpass_observations.py`.

Run cells in order. If the raw folder is empty, first run `python -m src.download_qpass` from the repository root. Original workbooks are never edited. Loader exclusions are retained in metadata; this audit covers rows returned by the loader.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.qpass_loader import load_all_qpass
from src.qpass_observations import (
    reshape_loaded_observations,
    validate_observations,
)

RAW_DIR = REPO_ROOT / "data" / "raw" / "QPASS"
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "qpass-notebook"


## 1. Load the supported workbooks

Dylan's loader combines the supported workbook formats into one DataFrame, preserving source file, sheet, row, academic year, and quarter. Counts remain in `open_8am`, `open_10am`, `open_12pm`, and `open_2pm`. Display its audit information as well: rows excluded by the loader do not reach task #2's checks.


In [2]:
loaded_data = load_all_qpass(RAW_DIR)
loader_audit = loaded_data.attrs.copy()
print("Workbooks loaded:", len(loader_audit["files"]))
print("Source rows returned by loader:", len(loaded_data))
display(loaded_data.head())
print("Loader exclusions (per file and sheet):")
display(loader_audit["dropped_rows"])


Workbooks loaded: 15
Source rows returned by loader: 23428
Loader exclusions (per file and sheet):


,source_file,sheet_name,source_row,format_version,academic_year,quarter,quarter_number,date,day,week,location,lot_or_structure,level,zone,source_order,report_area,neighborhood,type,sub_type,report_type,sub_type_in_format,report_type_in_format,total_stall_count,open_8am,open_10am,open_12pm,open_2pm,raw_structure,raw_lot
0,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,5,A_wide_weekly,2018-2019,Summer,1,2018-06-18,Monday,None,Athena,Structure,Level 1,NaN,NaN,NaN,NaN,Visitor,NaN,NaN,False,False,64,0,0,0,2,Athena,Level 1
1,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,6,A_wide_weekly,2018-2019,Summer,1,2018-06-18,Monday,None,Athena,Structure,Level 2,NaN,NaN,NaN,NaN,Visitor,NaN,NaN,False,False,155,118,0,1,12,NaN,Level 2
2,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,7,A_wide_weekly,2018-2019,Summer,1,2018-06-18,Monday,None,Athena,Structure,Level 3,NaN,NaN,NaN,NaN,A,NaN,NaN,False,False,40,19,2,6,4,NaN,Level 3
3,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,8,A_wide_weekly,2018-2019,Summer,1,2018-06-18,Monday,None,Athena,Structure,Level 3,NaN,NaN,NaN,NaN,B,NaN,NaN,False,False,103,0,0,0,0,NaN,NaN
4,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,9,A_wide_weekly,2018-2019,Summer,1,2018-06-18,Monday,None,Athena,Structure,Level 3,NaN,NaN,NaN,NaN,Visitor,NaN,NaN,False,False,39,12,0,2,4,NaN,NaN


{'QPASS 2018-2019-1 Summer.xlsx': {'1 Summer 2018 June 18': {'summary section (totals, not a parking location)': 16}, '2 Summer 2018 June 25th': {'summary section (totals, not a parking location)': 16}, '3 Summer 2018 July 2nd': {'summary section (totals, not a parking location)': 16}, '4 Summer 2018 July 9th': {'summary section (totals, not a parking location)': 16}}, 'QPASS 2018-2019-2 Fall.xlsx': {}, 'QPASS 2018-2019-3 Winter.xlsx': {}, 'QPASS 2018-2019-4 Spring.xlsx': {}, 'QPASS 2019-2020-1 Summer.xlsx': {'SS19 W1 Data': {'no counts and no total stall count': 3, 'summary section (totals, not a parking location)': 16}, 'SS19 W2 Data': {'no counts and no total stall count': 3, 'summary section (totals, not a parking location)': 16}}, 'QPASS 2019-2020-2 Fall.xlsx': {'Fall19 Week 1 Data': {'no counts and no total stall count': 6, 'summary section (totals, not a parking location)': 23}, 'Fall19 Week 2 Data': {'no counts and no total stall count': 6, 'summary section (totals, not a parki

## 2. Reshape and standardize values

`reshape_loaded_observations()` uses the shared loader's names. `location` is the facility, `type` is the category, and `sub_type` is the optional subtype. `report_type` is auxiliary metadata, not a replacement for `type`.

Each source row produces four time observations. Original identity fields remain unchanged; `raw_date`, `raw_total_stall_count`, and `raw_open_spaces` retain values before parsing. `date` and `observation_datetime` use local wall time; `total_stall_count` is the workbook count, not verified inventory capacity. See `docs/qpass_observation_contract.md` for downstream integration.


In [3]:
# These functions are imported from src/qpass_observations.py above.
print("Reshaping function comes from:", reshape_loaded_observations.__module__)


Reshaping function comes from: src.qpass_observations


## 3. Define the quality checks

Checks cover missing required identity fields, blank readings, unexpected nonnumeric text, negative/nonfinite/fractional counts, zero capacity, and open spaces above capacity. Whitespace-only readings count as blank. Text such as `FULL`, `n/a` or `-` is preserved and flagged as unexpected text; no numeric meaning is guessed.

Older Format A sheets never had `Report Type` or `Sub Type`. Their status is `not_in_format`, which does not fail required-field checks. The loader now records actual per-sheet header presence for both optional fields. Verified absent headers in any format get `not_in_format`; present headers with blank cells get `blank`. `blank_or_unavailable` is reserved for tables lacking header-presence evidence. `type` remains required in every format.

`exact_repeated_record` means the identity and capacity/count payload repeat. `ambiguous_allocation_requires_review` means a Format C identity has differing capacities, which may represent unnamed allocations. Other differing payloads get `conflicting_observation`. Within ambiguous groups, rows with the same identity and capacity but different open counts also retain `conflicting_observation`; other capacity subsets do not inherit that conflict. A row can have both flags when an exact repeated subset occurs inside a conflicting group. All records remain; `observation_group` groups related identities for review. Identity includes datetime, location, type, sub_type, and level, but excludes source-row provenance so repeats across sheets/files are detected.

Per-check counts overlap. `any_check` counts unique failed observations.


The validation function is imported above rather than redefined here. Open `src/qpass_observations.py` to read its implementation.

`validate_observations(observations)` returns two DataFrames: the observations with flags, and per-check pass/fail counts. Tests in `tests/test_qpass_observations.py` call this same function with small sample tables containing known problems.


## 4. Run reshaping and validation

The adapter reshapes every row returned by Dylan's loader into four time observations. Missing counts and partially filled observations are retained and flagged. Rows excluded by the loader are recorded separately in its audit metadata.


The connected workflow is:

```text
Excel workbooks -> load_all_qpass() [Dylan]
               -> reshape_loaded_observations() [Vivaan]
               -> validate_observations() -> results and exports
```

All workbook parsing uses Dylan's shared loader; the separate representative reader has been removed. Source file, sheet_name, source_row, format_version, year, quarter, level and subtype are retained. Loader exclusions remain in audit metadata and should be reviewed separately.


In [4]:
observations = reshape_loaded_observations(loaded_data)
audited, quality_summary = validate_observations(observations)
questionable_records = audited.loc[~audited["quality_pass"]].copy()

assert len(observations) == 4 * len(loaded_data)
assert len(audited) == len(observations)
print(f"Observation rows: {len(audited):,}")
print(f"Passing rows: {audited['quality_pass'].sum():,}")
print(f"Flagged rows: {len(questionable_records):,}")
display(quality_summary)
display(audited[["source_file", "sheet_name", "source_row", "observation_datetime",
                 "location", "type", "sub_type",
                 "total_stall_count", "open_spaces", "quality_flag"]].head(12))


Observation rows: 93,712
Passing rows: 77,336
Flagged rows: 16,376


,check,failed,passed
0,missing_observation_datetime,0,93712
1,missing_location,0,93712
2,missing_type,252,93460
3,missing_total_stall_count,72,93640
4,unexpected_text_total_stall_count,0,93712
5,negative_total_stall_count,0,93712
6,nonfinite_total_stall_count,0,93712
7,fractional_total_stall_count,0,93712
8,missing_open_spaces,5376,88336
9,unexpected_text_open_spaces,5,93707


,source_file,sheet_name,source_row,observation_datetime,location,type,sub_type,total_stall_count,open_spaces,quality_flag
0,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,5,2018-06-18 08:00:00,Athena,Visitor,NaN,64.0,0.0,pass
1,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,6,2018-06-18 08:00:00,Athena,Visitor,NaN,155.0,118.0,pass
2,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,7,2018-06-18 08:00:00,Athena,A,NaN,40.0,19.0,pass
3,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,8,2018-06-18 08:00:00,Athena,B,NaN,103.0,0.0,pass
4,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,9,2018-06-18 08:00:00,Athena,Visitor,NaN,39.0,12.0,pass
5,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,10,2018-06-18 08:00:00,Athena,A,NaN,122.0,66.0,pass
6,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,11,2018-06-18 08:00:00,Athena,B,NaN,96.0,0.0,pass
7,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,12,2018-06-18 08:00:00,Athena,B,NaN,224.0,26.0,pass
8,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,13,2018-06-18 08:00:00,Athena,B,NaN,156.0,94.0,pass
9,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,14,2018-06-18 08:00:00,Athena,B,NaN,100.0,6.0,pass


## 5. Examine questionable records

These records need review, not automatic deletion. Use the source file, sheet, and row to investigate the original values. The sample below displays up to five records per failed check; the full dataframe remains available as `questionable_records`.


In [5]:
audit_columns = ["source_file", "sheet_name", "source_row", "observation_group",
                 "observation_datetime", "location", "type", "sub_type", "level",
                 "raw_total_stall_count", "raw_open_spaces", "quality_flag"]
for check in quality_summary.loc[quality_summary["failed"].gt(0), "check"]:
    if check == "any_check":
        continue
    print(check)
    matching = audited["quality_flag"].str.split("|").map(lambda flags: check in flags)
    display(audited.loc[matching, audit_columns].head())

print("Optional source-field status (not failed observations):")
display(audited.groupby(["format_version", "report_type_status"], dropna=False).size())


missing_type
missing_total_stall_count
missing_open_spaces
unexpected_text_open_spaces
negative_open_spaces
fractional_open_spaces
zero_capacity
open_exceeds_capacity
exact_repeated_record
ambiguous_allocation_requires_review
conflicting_observation
Optional source-field status (not failed observations):


,source_file,sheet_name,source_row,observation_group,observation_datetime,location,type,sub_type,level,raw_total_stall_count,raw_open_spaces,quality_flag
1395,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,65,<NA>,2018-07-02 08:00:00,P017,NaN,NaN,NaN,258,NaN,missing_type|missing_open_spaces
1527,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,65,<NA>,2018-07-03 08:00:00,P017,NaN,NaN,NaN,258,242,missing_type
1659,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,65,<NA>,2018-07-04 08:00:00,P017,NaN,NaN,NaN,258,NaN,missing_type|missing_open_spaces
1791,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,65,<NA>,2018-07-05 08:00:00,P017,NaN,NaN,NaN,258,254,missing_type
1923,QPASS 2018-2019-1 Summer.xlsx,3 Summer 2018 July 2nd,65,<NA>,2018-07-06 08:00:00,P017,NaN,NaN,NaN,258,253,missing_type


,source_file,sheet_name,source_row,observation_group,observation_datetime,location,type,sub_type,level,raw_total_stall_count,raw_open_spaces,quality_flag
10255,QPASS 2024-2025-1 Summer.xlsm,QPASS Data Entry,107,<NA>,2024-07-01 08:00:00,Campus Point East,NaN,NaN,NaN,NaN,NaN,missing_type|missing_total_stall_count|missing_open_spaces
10364,QPASS 2024-2025-1 Summer.xlsm,QPASS Data Entry,216,<NA>,2024-07-02 08:00:00,Campus Point East,NaN,NaN,NaN,NaN,NaN,missing_type|missing_total_stall_count|missing_open_spaces
10473,QPASS 2024-2025-1 Summer.xlsm,QPASS Data Entry,325,<NA>,2024-07-03 08:00:00,Campus Point East,NaN,NaN,NaN,NaN,NaN,missing_type|missing_total_stall_count|missing_open_spaces
10582,QPASS 2024-2025-1 Summer.xlsm,QPASS Data Entry,434,<NA>,2024-07-04 08:00:00,Campus Point East,NaN,NaN,NaN,NaN,NaN,missing_type|missing_total_stall_count|missing_open_spaces
11564,QPASS 2024-2025-2 Fall.xlsm,QPASS Data Entry (Week 1),102,11559,2024-09-30 08:00:00,South,B Carpool,NaN,NaN,NaN,49,missing_total_stall_count


,source_file,sheet_name,source_row,observation_group,observation_datetime,location,type,sub_type,level,raw_total_stall_count,raw_open_spaces,quality_flag
38,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,43,39,2018-06-18 08:00:00,Pangea,B,NaN,Level 3,159,NaN,missing_open_spaces
39,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,44,40,2018-06-18 08:00:00,Pangea,S,NaN,Level 3,30,NaN,missing_open_spaces
42,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,47,43,2018-06-18 08:00:00,Pangea,V,NaN,Level 5,46,NaN,missing_open_spaces
171,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,43,172,2018-06-19 08:00:00,Pangea,B,NaN,Level 3,159,NaN,missing_open_spaces
172,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,44,173,2018-06-19 08:00:00,Pangea,S,NaN,Level 3,30,NaN,missing_open_spaces


,source_file,sheet_name,source_row,observation_group,observation_datetime,location,type,sub_type,level,raw_total_stall_count,raw_open_spaces,quality_flag
9008,QPASS 2019-2020-2 Fall.xlsx,Fall19 Week 1 Data,44,9004,2019-09-30 08:00:00,Campus Point East,B,NaN,NaN,242,-,unexpected_text_open_spaces
16080,QPASS 2025-2026-1 Summer.xlsm,QPASS Data Entry (Week 2),357,16075,2025-07-09 08:00:00,Gilman,A,NaN,NaN,401,`95,unexpected_text_open_spaces
32436,QPASS 2019-2020-2 Fall.xlsx,Fall19 Week 1 Data,44,31984,2019-09-30 10:00:00,Campus Point East,B,NaN,NaN,242,-,unexpected_text_open_spaces
55864,QPASS 2019-2020-2 Fall.xlsx,Fall19 Week 1 Data,44,54964,2019-09-30 12:00:00,Campus Point East,B,NaN,NaN,242,-,unexpected_text_open_spaces
79292,QPASS 2019-2020-2 Fall.xlsx,Fall19 Week 1 Data,44,77944,2019-09-30 14:00:00,Campus Point East,B,NaN,NaN,242,-,unexpected_text_open_spaces


...

,source_file,sheet_name,source_row,observation_group,observation_datetime,location,type,sub_type,level,raw_total_stall_count,raw_open_spaces,quality_flag
40,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,45,41,2018-06-18 08:00:00,Pangea,B,NaN,Level 4,77,102,open_exceeds_capacity
173,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,45,174,2018-06-19 08:00:00,Pangea,B,NaN,Level 4,77,85,open_exceeds_capacity
433,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,39,434,2018-06-21 08:00:00,Hopkins,S,NaN,Level 7,58,65,open_exceeds_capacity
572,QPASS 2018-2019-1 Summer.xlsx,1 Summer 2018 June 18,45,573,2018-06-22 08:00:00,Pangea,B,NaN,Level 4,77,80,open_exceeds_capacity
734,QPASS 2018-2019-1 Summer.xlsx,2 Summer 2018 June 25th,74,735,2018-06-25 08:00:00,P206,A,NaN,NaN,57,60,open_exceeds_capacity


,source_file,sheet_name,source_row,observation_group,observation_datetime,location,type,sub_type,level,raw_total_stall_count,raw_open_spaces,quality_flag
11348,QPASS 2024-2025-2 Fall.xlsm,QPASS Data Entry (Week 0),110,11344,2024-09-26 08:00:00,Theatre District Garage,S,NaN,NaN,50,NaN,missing_open_spaces|exact_repeated_record
11349,QPASS 2024-2025-2 Fall.xlsm,QPASS Data Entry (Week 0),111,11344,2024-09-26 08:00:00,Theatre District Garage,S,NaN,NaN,50,NaN,missing_open_spaces|exact_repeated_record
16492,QPASS 2025-2026-2 Fall.xlsm,QPASS Data Entry (Week 0),64,16487,2025-09-25 08:00:00,P507,Visitor,NaN,NaN,0,0,zero_capacity|exact_repeated_record
16493,QPASS 2025-2026-2 Fall.xlsm,QPASS Data Entry (Week 0),65,16487,2025-09-25 08:00:00,P507,Visitor,NaN,NaN,0,0,zero_capacity|exact_repeated_record
16631,QPASS 2025-2026-2 Fall.xlsm,QPASS Data Entry (Week 0),214,16625,2025-09-26 08:00:00,P507,Visitor,NaN,NaN,0,0,zero_capacity|exact_repeated_record


,source_file,sheet_name,source_row,observation_group,observation_datetime,location,type,sub_type,level,raw_total_stall_count,raw_open_spaces,quality_flag
18109,QPASS 2025-2026-4 Spring.xlsm,Data Entry,8,18092,2026-03-30 08:00:00,8980,Visitor,NaN,NaN,2,2,ambiguous_allocation_requires_review
18110,QPASS 2025-2026-4 Spring.xlsm,Data Entry,9,18092,2026-03-30 08:00:00,8980,Visitor,NaN,NaN,53,41,ambiguous_allocation_requires_review
18113,QPASS 2025-2026-4 Spring.xlsm,Data Entry,12,18095,2026-03-30 08:00:00,Athena,Visitor,NaN,NaN,30,16,ambiguous_allocation_requires_review
18114,QPASS 2025-2026-4 Spring.xlsm,Data Entry,13,18095,2026-03-30 08:00:00,Athena,Visitor,NaN,NaN,9,2,ambiguous_allocation_requires_review
18115,QPASS 2025-2026-4 Spring.xlsm,Data Entry,14,18095,2026-03-30 08:00:00,Athena,Visitor,NaN,NaN,10,0,ambiguous_allocation_requires_review


,source_file,sheet_name,source_row,observation_group,observation_datetime,location,type,sub_type,level,raw_total_stall_count,raw_open_spaces,quality_flag
8504,QPASS 2019-2020-1 Summer.xlsx,SS19 W2 Data,100,8505,2019-07-08 08:00:00,Pangea,V,NaN,NaN,145,136,conflicting_observation
8505,QPASS 2019-2020-1 Summer.xlsx,SS19 W2 Data,101,8505,2019-07-08 08:00:00,Pangea,V,NaN,NaN,9,5,conflicting_observation
8616,QPASS 2019-2020-1 Summer.xlsx,SS19 W2 Data,100,8616,2019-07-09 08:00:00,Pangea,V,NaN,NaN,145,117,conflicting_observation
8617,QPASS 2019-2020-1 Summer.xlsx,SS19 W2 Data,101,8616,2019-07-09 08:00:00,Pangea,V,NaN,NaN,9,8,conflicting_observation
8728,QPASS 2019-2020-1 Summer.xlsx,SS19 W2 Data,100,8727,2019-07-10 08:00:00,Pangea,V,NaN,NaN,145,140,conflicting_observation


format_version  report_type_status
A_wide_weekly   not_in_format         40604
B_long_weekly   blank                    20
                present               31792
C_long_single   present               21296
dtype: int64

## 6. Export the deliverables

This cell writes generated CSV reports locally under `data/processed/qpass-notebook`. It does not edit the workbook or publish to GitHub. CSV readers should parse `observation_datetime` and `date` as dates when reloading.


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
audited.to_csv(OUTPUT_DIR / "observations.csv", index=False)
questionable_records.to_csv(OUTPUT_DIR / "questionable_records.csv", index=False)
quality_summary.to_csv(OUTPUT_DIR / "quality_summary.csv", index=False)
run_metadata = {
    "source_files": loader_audit["files"],
    "loader_audit": loader_audit,
    "observation_rows": len(audited),
    "passing_rows": int(audited["quality_pass"].sum()),
    "flagged_rows": len(questionable_records),
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
print("Reports saved to:", OUTPUT_DIR.resolve())


Reports saved to: C:\Users\vivpa\Documents\parksmart\data\processed\qpass-notebook


## Quality-flag definitions and unresolved questions

| Flag | Meaning |
| --- | --- |
| `missing_<identity>` | Required datetime, location or type is blank/unparseable |
| `missing_<count>` | Original reading is blank, including whitespace-only cells |
| `unexpected_text_<count>` | Nonblank reading cannot be parsed numerically |
| `negative_<count>` | Count below zero, with original precision retained |
| `nonfinite_<count>` | Infinite numeric value |
| `fractional_<count>` | Finite count is not an integer |
| `zero_capacity` | Workbook total_stall_count is zero |
| `open_exceeds_capacity` | Open count exceeds workbook total_stall_count |
| `exact_repeated_record` | Same complete identity and capacity/count payload |
| `ambiguous_allocation_requires_review` | Format C identity with differing capacities; allocation scope unverified |
| `conflicting_observation` | Other identity groups with different payloads |
| `pass` | No implemented checks fail |

- Report Type is optional metadata; type is the category. No category mapping is guessed.
- Optional fields marked not_in_format are structurally absent, not failed required values.
- Review exact repeats and conflicting groups before any deduplication or aggregation.
- Verify capacities with historical inventory; missing/text counts are not zero.
- Extremely small negative numbers may reflect floating-point calculations. Values remain unchanged pending review.
- This audit does not verify permits, entrances, or forecast reliability.
- The contract matches the loader names consumed by the remote label/capacity code; Daira's `prepare_label_records()` is on main and integration-tested with audited observations; Hanrong's capacity entry point still needs adaptation. See the contract document.


## Combined audit results - 2026-10-20 (PR #9 revisions)

Fresh execution of all code cells with the shared loader. All records remain available for review.

- Workbooks: **15**.
- Loader rows: **23,428**.
- Timed observations: **93,712**.
- Passing: **77,336**.
- Flagged: **16,376**.

| Check | Failed | Passed |
| --- | ---: | ---: |
| `missing_observation_datetime` | 0 | 93,712 |
| `missing_location` | 0 | 93,712 |
| `missing_type` | 252 | 93,460 |
| `missing_total_stall_count` | 72 | 93,640 |
| `unexpected_text_total_stall_count` | 0 | 93,712 |
| `negative_total_stall_count` | 0 | 93,712 |
| `nonfinite_total_stall_count` | 0 | 93,712 |
| `fractional_total_stall_count` | 0 | 93,712 |
| `missing_open_spaces` | 5,376 | 88,336 |
| `unexpected_text_open_spaces` | 5 | 93,707 |
| `negative_open_spaces` | 1 | 93,711 |
| `nonfinite_open_spaces` | 0 | 93,712 |
| `fractional_open_spaces` | 60 | 93,652 |
| `zero_capacity` | 8,052 | 85,660 |
| `open_exceeds_capacity` | 5,442 | 88,270 |
| `exact_repeated_record` | 80 | 93,632 |
| `ambiguous_allocation_requires_review` | 3,240 | 90,472 |
| `conflicting_observation` | 64 | 93,648 |
| `any_check` | 16,376 | 77,336 |

Format C capacity variants are ambiguous allocations requiring review; no capacity is selected or summed.
Source review: Spring Athena Visitor rows 12-15 have capacities 30, 9, 10 and 210.
Zone, Order and Report Area are all 0 and spatial/report labels match. No verified allocation identifier was found.
`source_order` is preserved for review. See the observation contract for the source inspection details.

Optional statuses (informational):

| Format | Field | Status | Rows |
| --- | --- | --- | ---: |
| A_wide_weekly | sub_type_status | not_in_format | 40,604 |
| B_long_weekly | sub_type_status | not_in_format | 31,812 |
| C_long_single | sub_type_status | blank | 4,704 |
| C_long_single | sub_type_status | not_in_format | 13,280 |
| C_long_single | sub_type_status | present | 3,312 |
| A_wide_weekly | report_type_status | not_in_format | 40,604 |
| B_long_weekly | report_type_status | blank | 20 |
| B_long_weekly | report_type_status | present | 31,792 |
| C_long_single | report_type_status | present | 21,296 |

Examples retained with source provenance:

**ambiguous_allocation_requires_review**

| File | Row | Location | Type | Capacity | Open |
| --- | ---: | --- | --- | ---: | ---: |
| QPASS 2025-2026-4 Spring.xlsm | 8 | 8980 | Visitor | 2.0 | 2.0 |
| QPASS 2025-2026-4 Spring.xlsm | 9 | 8980 | Visitor | 53.0 | 41.0 |
| QPASS 2025-2026-4 Spring.xlsm | 12 | Athena | Visitor | 30.0 | 16.0 |
| QPASS 2025-2026-4 Spring.xlsm | 13 | Athena | Visitor | 9.0 | 2.0 |
| QPASS 2025-2026-4 Spring.xlsm | 14 | Athena | Visitor | 10.0 | 0.0 |

**conflicting_observation**

| File | Row | Location | Type | Capacity | Open |
| --- | ---: | --- | --- | ---: | ---: |
| QPASS 2019-2020-1 Summer.xlsx | 100 | Pangea | V | 145.0 | 136.0 |
| QPASS 2019-2020-1 Summer.xlsx | 101 | Pangea | V | 9.0 | 5.0 |
| QPASS 2019-2020-1 Summer.xlsx | 100 | Pangea | V | 145.0 | 117.0 |
| QPASS 2019-2020-1 Summer.xlsx | 101 | Pangea | V | 9.0 | 8.0 |
| QPASS 2019-2020-1 Summer.xlsx | 100 | Pangea | V | 145.0 | 140.0 |

**exact_repeated_record**

| File | Row | Location | Type | Capacity | Open |
| --- | ---: | --- | --- | ---: | ---: |
| QPASS 2024-2025-2 Fall.xlsm | 110 | Theatre District Garage | S | 50.0 | <NA> |
| QPASS 2024-2025-2 Fall.xlsm | 111 | Theatre District Garage | S | 50.0 | <NA> |
| QPASS 2025-2026-2 Fall.xlsm | 64 | P507 | Visitor | 0.0 | 0.0 |
| QPASS 2025-2026-2 Fall.xlsm | 65 | P507 | Visitor | 0.0 | 0.0 |
| QPASS 2025-2026-2 Fall.xlsm | 214 | P507 | Visitor | 0.0 | 0.0 |

